# 바람(Wind) Robustness 실험 — Colab Runner (Google Drive 방식)

`comm_env.py + comm_train.py + comm_eval.py + wind_sweep.py` 바람 외란 robustness 실험용 노트북.

GitHub 없이 **Drive에 올린 프로젝트 폴더**에서 직접 실행하는 버전.

**먼저 `런타임 → 런타임 유형 변경 → GPU`를 선택하세요.**

### 사전 준비
`내 드라이브`에 `RL-2026s1-tp` 폴더를 만들고 아래 파일을 업로드해 둘 것:
`comm_env.py, comm_train.py, comm_eval.py, formation_seq.py, wind_sweep.py, test_wind.py`

실험 구성: **GROUND→D→G (2단계), 드론 14대**
- 정책 A (baseline): 바람 없이 학습
- 정책 B (robust): 바람 도메인 랜덤화 [0, 0.3]로 학습

## 1. Google Drive 마운트 + 작업 폴더로 이동

실행하면 인증 팝업 → 계정 승인. `!ls` 출력에 `.py` 파일들이 보이면 성공.
폴더 이름이 다르면 아래 경로를 수정하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/RL-2026s1-tp
!ls

## 2. 패키지 설치

In [ ]:
!pip install -q torchrl pettingzoo==1.24.3 gymnasium scipy matplotlib pillow tensorboard

## 3. 코드 반영 확인

출력 시그니처에 `n_agents`, `wind_prob, wind_strength, wind_dir, randomize_wind`가 보이면 정상.

In [ ]:
import inspect, comm_env
print(inspect.signature(comm_env.ShapeFormationEnv))

## 4. 바람 기능 스모크 테스트

3가지 검증 (바람 밀기 / 도메인 랜덤화 / 바람 끄면 원래대로). 모두 OK여야 함.

In [ ]:
!python test_wind.py

## 5. 빠른 파이프라인 검증

본 학습 전, GROUND→X 소규모 실행으로 정상 동작만 확인 (수 분). reward가 오르는지만 확인.

In [ ]:
!python comm_train.py --shapes GROUND,X --max-steps 250 --total-frames 50000 --frames-per-batch 4096 --wind-prob 0.3 --randomize-wind --ckpt-every 5 --save-dir ckpt_smoke_wind --tb-logdir runs_smoke_wind

## 6. 정책 A — 베이스라인 (바람 없음)

GROUND→D→G 를 바람 없이 학습. GPU 필수. 체크포인트는 Drive 폴더 안에 저장됨.

In [ ]:
!python comm_train.py --shapes "GROUND,D,G" --max-steps 220 --total-frames 800000 --frames-per-batch 4096 --ckpt-every 10 --save-dir ckpt_dg_base --tb-logdir runs_dg_base

## 7. 정책 B — 강인 (바람 도메인 랜덤화 [0, 0.3])

`--wind-prob 0.3 --randomize-wind` → 매 에피소드 바람 세기 [0, 0.3]·방향 랜덤.

In [ ]:
!python comm_train.py --shapes "GROUND,D,G" --max-steps 220 --total-frames 800000 --frames-per-batch 4096 --wind-prob 0.3 --randomize-wind --ckpt-every 10 --save-dir ckpt_dg_robust --tb-logdir runs_dg_robust

### (선택) 세션이 끊겼다면 — 이어서 학습

`--load-ckpt`로 마지막 체크포인트에서 재개. 덮어쓰기 방지를 위해 save-dir을 새로 지정. `ckpt_100.pt`는 실제 마지막 번호로 바꿀 것.

In [ ]:
!python comm_train.py --shapes "GROUND,D,G" --max-steps 220 --total-frames 800000 --frames-per-batch 4096 --wind-prob 0.3 --randomize-wind --load-ckpt ckpt_dg_robust/ckpt_100.pt --ckpt-every 10 --save-dir ckpt_dg_robust_resumed --tb-logdir runs_dg_robust

## 8. TensorBoard

학습 진행 중 mean_episode_reward·success_rate 곡선 확인.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs_dg_robust

## 9. 바람 스윕 비교 — robustness 곡선

정책 A·B를 바람 세기별로 평가 → 성공률·커버리지 곡선 PNG 생성.

`ckpt_190.pt`는 실제 생성된 마지막 체크포인트 번호로 바꿀 것 (total_frames ÷ frames_per_batch ÷ ckpt_every).

In [ ]:
!python wind_sweep.py --ckpt-a ckpt_dg_base/ckpt_190.pt --ckpt-b ckpt_dg_robust/ckpt_190.pt --shapes "GROUND,D,G" --max-steps 220 --wind-levels 0.0,0.1,0.2,0.3,0.4 --n-episodes 100 --out wind_sweep_dg.png

In [ ]:
from IPython.display import Image
Image("wind_sweep_dg.png")

## 10. 바람 속 드론쇼 GIF + 평가

강인 정책 B를 바람 0.3에서 평가하고 GIF 저장.

In [ ]:
!python comm_eval.py --ckpt ckpt_dg_robust/ckpt_190.pt --shapes "GROUND,D,G" --max-steps 220 --wind-prob 0.3 --greedy --n-episodes 50 --save-gif dg_wind.gif --out eval_dg_wind.txt

In [ ]:
from IPython.display import Image
Image("dg_wind.gif")